# CSQA Fixed-Site Rescue Patching And Activation Steering Analysis

Intervention notebook for rescue-site validation with a single late-layer intervention location.

- model family: Qwen2.5
- target checkpoint: 3B instruct model
- corruption type: additive noise on one earlier decoder layer output
- patch type: same-example clean activation replacement at one fixed rescue layer
- steering types:
  - contrastive mean direction
  - probe-normal direction
- decision space: constrained A-E answer-choice logits only


In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

from src.data.load_csqa import load_csqa

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)


## Configuration

RESCUE_LAYER_1BASED = None resolves to the final decoder layer. CORRUPTION_LAYERS_1BASED = None resolves to all earlier decoder layers.


In [ ]:
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
TRAIN_SPLIT = "train"
EVAL_SPLIT = "validation"
TRAIN_LIMIT = 512
EVAL_LIMIT = 32
MAX_SEQ_LEN = 384
CLEAN_BATCH_SIZE = 4
INTERVENTION_BATCH_SIZE = 2
CORRUPTION_NOISE_SCALES = [0.10, 0.25, 0.50]
RESCUE_LAYER_1BASED = None
CORRUPTION_LAYERS_1BASED = None
STEERING_SCALES = [1.0]
PROBE_EPOCHS = 200
PROBE_LR = 5e-2
PROBE_WEIGHT_DECAY = 1e-4
SEED = 42
USE_BFLOAT16_IF_AVAILABLE = True


## Environment Setup


In [ ]:
torch.manual_seed(SEED)
np.random.seed(SEED)

LETTERS = ["A", "B", "C", "D", "E"]

train_rows = load_csqa(split=TRAIN_SPLIT, limit=TRAIN_LIMIT).copy()
eval_rows = load_csqa(split=EVAL_SPLIT, limit=EVAL_LIMIT).copy()

for frame in [train_rows, eval_rows]:
    frame["n_choices"] = frame["csqa_choices"].map(len)
    frame["prompt_len_chars"] = frame["text"].str.len()
    assert frame["n_choices"].eq(5).all(), "Expected 5 choices for every CSQA row."

if torch.cuda.is_available():
    if USE_BFLOAT16_IF_AVAILABLE and torch.cuda.is_bf16_supported():
        model_dtype = torch.bfloat16
    else:
        model_dtype = torch.float16
    device_map = "auto"
else:
    model_dtype = torch.float32
    device_map = None

tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=model_dtype,
    device_map=device_map,
    attn_implementation="eager",
)
model.eval()

input_device = model.device if hasattr(model, "device") else next(model.parameters()).device


def build_answer_token_ids(tok):
    out = {}
    for letter in LETTERS:
        ids = tok(" " + letter, add_special_tokens=False)["input_ids"]
        if len(ids) != 1:
            raise ValueError(f"Answer token '{letter}' is not single-token: {ids}")
        out[letter] = int(ids[0])
    return out


answer_token_ids = build_answer_token_ids(tok)
answer_ids = [answer_token_ids[l] for l in LETTERS]
answer_id_tensor = torch.tensor(answer_ids, dtype=torch.long)

display(eval_rows[["example_id", "answerKey", "prompt_len_chars"]].head())
print("train rows:", len(train_rows))
print("eval rows:", len(eval_rows))
print("answer token ids:", answer_token_ids)


## Helper Functions


In [ ]:
def get_decoder_layers(model):
    candidates = [
        "model.layers",
        "transformer.h",
        "gpt_neox.layers",
    ]
    for path in candidates:
        cur = model
        ok = True
        for part in path.split("."):
            if not hasattr(cur, part):
                ok = False
                break
            cur = getattr(cur, part)
        if ok:
            return cur
    raise ValueError("Could not locate decoder layers on this model.")


def encode_prompts(texts, tok, max_seq_len):
    batch = tok(
        list(texts),
        add_special_tokens=False,
        truncation=True,
        max_length=max_seq_len,
        padding=True,
        return_tensors="pt",
    )
    pos = []
    for mask in batch["attention_mask"]:
        nz = torch.nonzero(mask, as_tuple=False).view(-1)
        pos.append(int(nz[-1].item()))
    batch["decision_pos"] = torch.tensor(pos, dtype=torch.long)
    return batch


def unpack_output_hidden(output):
    if isinstance(output, tuple):
        return output[0]
    return output


def repack_output_hidden(output, new_hidden):
    if isinstance(output, tuple):
        return (new_hidden,) + tuple(output[1:])
    return new_hidden


def select_choice_logits(logits, decision_pos):
    row_idx = torch.arange(logits.shape[0], device=logits.device)
    answer_ids_on_device = answer_id_tensor.to(logits.device)
    return logits[row_idx, decision_pos][:, answer_ids_on_device].float()


def compute_choice_metrics(choice_logits, true_choice_idx):
    pred_idx = choice_logits.argmax(dim=-1)
    row_idx = torch.arange(choice_logits.shape[0], device=choice_logits.device)
    true_logits = choice_logits[row_idx, true_choice_idx]
    other_logits = choice_logits.clone()
    other_logits[row_idx, true_choice_idx] = -torch.inf
    best_other = other_logits.max(dim=-1).values
    correct_margin = true_logits - best_other
    return pred_idx.detach().cpu(), correct_margin.detach().cpu()


def apply_token_noise(hidden, decision_pos, noise_scale):
    row_idx = torch.arange(hidden.shape[0], device=hidden.device)
    token_hidden = hidden[row_idx, decision_pos]
    rms = token_hidden.float().pow(2).mean(dim=-1, keepdim=True).sqrt().to(token_hidden.dtype)
    noise = torch.randn_like(token_hidden) * (noise_scale * rms)
    hidden_out = hidden.clone()
    hidden_out[row_idx, decision_pos] = token_hidden + noise
    return hidden_out


def patch_token_hidden(hidden, decision_pos, clean_hidden):
    row_idx = torch.arange(hidden.shape[0], device=hidden.device)
    hidden_out = hidden.clone()
    hidden_out[row_idx, decision_pos] = clean_hidden.to(hidden.device, dtype=hidden.dtype)
    return hidden_out


def apply_token_steering(hidden, decision_pos, direction, steering_scale):
    row_idx = torch.arange(hidden.shape[0], device=hidden.device)
    token_hidden = hidden[row_idx, decision_pos]
    rms = token_hidden.float().pow(2).mean(dim=-1, keepdim=True).sqrt().to(token_hidden.dtype)
    direction = direction.to(hidden.device, dtype=hidden.dtype)
    hidden_out = hidden.clone()
    hidden_out[row_idx, decision_pos] = token_hidden + (steering_scale * rms) * direction.unsqueeze(0)
    return hidden_out


def set_all_seeds(seed):
    torch.manual_seed(int(seed))
    np.random.seed(int(seed) % (2**32 - 1))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))


decoder_layers = get_decoder_layers(model)
L = len(decoder_layers)

resolved_rescue_layer_1based = L if RESCUE_LAYER_1BASED is None else int(RESCUE_LAYER_1BASED)
if not (1 <= resolved_rescue_layer_1based <= L):
    raise ValueError("RESCUE_LAYER_1BASED is out of range.")

if CORRUPTION_LAYERS_1BASED is None:
    resolved_corruption_layers_1based = list(range(1, resolved_rescue_layer_1based))
else:
    resolved_corruption_layers_1based = [int(x) for x in CORRUPTION_LAYERS_1BASED]

resolved_corruption_layers_1based = sorted(set(resolved_corruption_layers_1based))
for x in resolved_corruption_layers_1based:
    if not (1 <= x < resolved_rescue_layer_1based):
        raise ValueError("Every corruption layer must be earlier than the rescue layer.")

if len(resolved_corruption_layers_1based) == 0:
    raise ValueError("At least one earlier corruption layer is required.")

resolved_rescue_layer_0based = resolved_rescue_layer_1based - 1
resolved_corruption_layers_0based = [x - 1 for x in resolved_corruption_layers_1based]

print("decoder layers:", L)
print("resolved rescue layer:", resolved_rescue_layer_1based)
print("resolved corruption layers:", resolved_corruption_layers_1based[:12], "..." if len(resolved_corruption_layers_1based) > 12 else "")


## Data Generation And Extraction


### Clean Rescue-Layer Feature Extraction

Clean decoder layer outputs are extracted at the fixed rescue layer for steering direction construction and evaluation baselines.


In [ ]:
def extract_clean_layer_features(frame, target_layer_0based, batch_size, desc):
    rows = []
    hidden_blocks = []
    target_module = decoder_layers[target_layer_0based]

    for start in tqdm(range(0, len(frame), batch_size), total=int(math.ceil(len(frame) / batch_size)), desc=desc):
        batch_df = frame.iloc[start:start + batch_size].reset_index(drop=True)
        batch_cpu = encode_prompts(batch_df["text"], tok, MAX_SEQ_LEN)
        decision_pos = batch_cpu.pop("decision_pos")
        batch = {k: v.to(input_device) for k, v in batch_cpu.items()}
        decision_pos = decision_pos.to(input_device)
        true_choice_idx = torch.tensor([LETTERS.index(str(x)) for x in batch_df["answerKey"].tolist()], dtype=torch.long, device=input_device)

        captured = {}

        def capture_hook(module, inputs, output):
            captured["hidden"] = unpack_output_hidden(output).detach()

        handle = target_module.register_forward_hook(capture_hook)
        try:
            with torch.inference_mode():
                out = model(**batch, return_dict=True, use_cache=False)
        finally:
            handle.remove()

        choice_logits = select_choice_logits(out.logits, decision_pos)
        pred_idx, correct_margin = compute_choice_metrics(choice_logits, true_choice_idx)
        clean_is_correct = pred_idx.eq(true_choice_idx.detach().cpu())

        row_idx = torch.arange(len(batch_df), device=decision_pos.device)
        hidden = captured["hidden"][row_idx, decision_pos].detach().cpu().to(torch.float32)
        hidden_blocks.append(hidden)

        for i, row in batch_df.iterrows():
            rows.append(
                {
                    "example_id": row["example_id"],
                    "answerKey": row["answerKey"],
                    "clean_prediction_idx": int(pred_idx[i].item()),
                    "clean_correct_margin": float(correct_margin[i].item()),
                    "clean_is_correct": bool(clean_is_correct[i].item()),
                }
            )

    return pd.DataFrame(rows), torch.cat(hidden_blocks, dim=0)


train_clean_df, train_rescue_hidden = extract_clean_layer_features(
    train_rows,
    resolved_rescue_layer_0based,
    CLEAN_BATCH_SIZE,
    desc="train rescue-layer extraction",
)

eval_clean_df, eval_rescue_hidden = extract_clean_layer_features(
    eval_rows,
    resolved_rescue_layer_0based,
    CLEAN_BATCH_SIZE,
    desc="eval rescue-layer extraction",
)

clean_summary = pd.DataFrame(
    [
        {
            "split": "train",
            "n_examples": len(train_clean_df),
            "clean_accuracy": float(train_clean_df["clean_is_correct"].mean()),
            "mean_correct_margin": float(train_clean_df["clean_correct_margin"].mean()),
        },
        {
            "split": "validation",
            "n_examples": len(eval_clean_df),
            "clean_accuracy": float(eval_clean_df["clean_is_correct"].mean()),
            "mean_correct_margin": float(eval_clean_df["clean_correct_margin"].mean()),
        },
    ]
)

display(clean_summary.round(4))


### Steering Direction Construction

Two additive steering directions are constructed at the fixed rescue layer:

- contrastive mean direction: mean(correct) - mean(incorrect)
- probe-normal direction: linear probe normal for clean correctness classification


In [ ]:
def build_contrastive_mean_direction(hidden_cache, is_correct_mask):
    mask = torch.as_tensor(is_correct_mask.to_numpy() if hasattr(is_correct_mask, "to_numpy") else is_correct_mask, dtype=torch.bool)
    if int(mask.sum().item()) == 0 or int((~mask).sum().item()) == 0:
        raise ValueError("Both correct and incorrect examples are required for contrastive mean direction construction.")

    correct_mean = hidden_cache[mask].mean(dim=0)
    incorrect_mean = hidden_cache[~mask].mean(dim=0)
    raw_direction = (correct_mean - incorrect_mean).to(torch.float32)
    raw_norm = float(raw_direction.norm().item())
    direction = raw_direction / raw_direction.norm().clamp_min(1e-12)
    projection = hidden_cache @ direction

    return direction, {
        "steering_method": "Contrastive mean direction",
        "n_examples": int(hidden_cache.shape[0]),
        "n_correct": int(mask.sum().item()),
        "n_incorrect": int((~mask).sum().item()),
        "raw_direction_norm": raw_norm,
        "mean_projection_correct": float(projection[mask].mean().item()),
        "mean_projection_incorrect": float(projection[~mask].mean().item()),
        "probe_train_accuracy": np.nan,
    }


def build_probe_normal_direction(hidden_cache, is_correct_mask):
    mask = torch.as_tensor(is_correct_mask.to_numpy() if hasattr(is_correct_mask, "to_numpy") else is_correct_mask, dtype=torch.bool)
    if int(mask.sum().item()) == 0 or int((~mask).sum().item()) == 0:
        raise ValueError("Both correct and incorrect examples are required for probe direction construction.")

    x = hidden_cache.to(torch.float32)
    y = mask.to(torch.float32).unsqueeze(-1)
    mu = x.mean(dim=0)
    sigma = x.std(dim=0).clamp_min(1e-6)
    xz = (x - mu) / sigma

    probe = torch.nn.Linear(xz.shape[1], 1)
    optimizer = torch.optim.AdamW(probe.parameters(), lr=PROBE_LR, weight_decay=PROBE_WEIGHT_DECAY)

    for _ in tqdm(range(PROBE_EPOCHS), desc="probe training"):
        optimizer.zero_grad(set_to_none=True)
        logits = probe(xz)
        loss = F.binary_cross_entropy_with_logits(logits, y)
        loss.backward()
        optimizer.step()

    with torch.inference_mode():
        logits = probe(xz)
        preds = logits.squeeze(-1).ge(0.0)
        train_accuracy = float(preds.eq(mask).float().mean().item())
        raw_weight = probe.weight.detach().cpu().squeeze(0).to(torch.float32)

    raw_direction = raw_weight / sigma.cpu()
    projection = x @ raw_direction
    if projection[mask].mean() < projection[~mask].mean():
        raw_direction = -raw_direction
        projection = -projection

    raw_norm = float(raw_direction.norm().item())
    direction = raw_direction / raw_direction.norm().clamp_min(1e-12)

    return direction, {
        "steering_method": "Probe-normal direction",
        "n_examples": int(x.shape[0]),
        "n_correct": int(mask.sum().item()),
        "n_incorrect": int((~mask).sum().item()),
        "raw_direction_norm": raw_norm,
        "mean_projection_correct": float(projection[mask].mean().item()),
        "mean_projection_incorrect": float(projection[~mask].mean().item()),
        "probe_train_accuracy": train_accuracy,
    }


contrastive_direction, contrastive_info = build_contrastive_mean_direction(
    train_rescue_hidden,
    train_clean_df["clean_is_correct"],
)

probe_direction, probe_info = build_probe_normal_direction(
    train_rescue_hidden,
    train_clean_df["clean_is_correct"],
)

steering_direction_info_df = pd.DataFrame([contrastive_info, probe_info])
steering_directions = {
    "Contrastive mean direction": contrastive_direction,
    "Probe-normal direction": probe_direction,
}

display(steering_direction_info_df.round(4))


### Fixed-Site Corruption, Same-Example Clean Patching, And Steering

Each earlier corruption site is scanned one at a time across multiple configured corruption scales. The rescue site is fixed. Three intervention regimes are evaluated:

- corruption only
- same-example clean patch at the rescue layer
- additive steering at the rescue layer


In [ ]:
def run_fixed_site_intervention_scan(
    frame,
    corruption_layers_0based,
    rescue_layer_0based,
    noise_scale,
    batch_size,
    intervention_kind,
    clean_rescue_hidden=None,
    steering_direction=None,
    steering_scale=None,
    desc="fixed-site intervention",
):
    rows = []
    rescue_module = decoder_layers[rescue_layer_0based]

    for corruption_layer_0based in tqdm(corruption_layers_0based, desc=desc):
        corruption_module = decoder_layers[corruption_layer_0based]

        for start in range(0, len(frame), batch_size):
            batch_df = frame.iloc[start:start + batch_size].reset_index(drop=True)
            batch_cpu = encode_prompts(batch_df["text"], tok, MAX_SEQ_LEN)
            decision_pos = batch_cpu.pop("decision_pos")
            batch = {k: v.to(input_device) for k, v in batch_cpu.items()}
            decision_pos = decision_pos.to(input_device)
            true_choice_idx = torch.tensor([LETTERS.index(str(x)) for x in batch_df["answerKey"].tolist()], dtype=torch.long, device=input_device)

            local_seed = int(SEED + 10007 * (corruption_layer_0based + 1) + 97 * start)
            set_all_seeds(local_seed)
            handles = []

            def corruption_hook(module, inputs, output):
                hidden = unpack_output_hidden(output)
                hidden = apply_token_noise(hidden, decision_pos, noise_scale)
                return repack_output_hidden(output, hidden)

            handles.append(corruption_module.register_forward_hook(corruption_hook))

            if intervention_kind == "clean_patch":
                batch_clean_hidden = clean_rescue_hidden[start:start + len(batch_df)]

                def patch_hook(module, inputs, output):
                    hidden = unpack_output_hidden(output)
                    hidden = patch_token_hidden(hidden, decision_pos, batch_clean_hidden)
                    return repack_output_hidden(output, hidden)

                handles.append(rescue_module.register_forward_hook(patch_hook))

            elif intervention_kind == "steering":
                if steering_direction is None or steering_scale is None:
                    raise ValueError("steering_direction and steering_scale are required for steering scans.")

                def steering_hook(module, inputs, output):
                    hidden = unpack_output_hidden(output)
                    hidden = apply_token_steering(hidden, decision_pos, steering_direction, steering_scale)
                    return repack_output_hidden(output, hidden)

                handles.append(rescue_module.register_forward_hook(steering_hook))

            try:
                with torch.inference_mode():
                    out = model(**batch, return_dict=True, use_cache=False)
            finally:
                for handle in handles:
                    handle.remove()

            choice_logits = select_choice_logits(out.logits, decision_pos)
            pred_idx, correct_margin = compute_choice_metrics(choice_logits, true_choice_idx)
            is_correct = pred_idx.eq(true_choice_idx.detach().cpu())

            for i, row in batch_df.iterrows():
                rows.append(
                    {
                        "example_id": row["example_id"],
                        "corruption_layer": int(corruption_layer_0based),
                        "prediction_idx": int(pred_idx[i].item()),
                        "correct_margin": float(correct_margin[i].item()),
                        "is_correct": bool(is_correct[i].item()),
                    }
                )

    return pd.DataFrame(rows)


corrupted_frames = []
for corruption_noise_scale in CORRUPTION_NOISE_SCALES:
    corrupted_part = run_fixed_site_intervention_scan(
        eval_rows,
        resolved_corruption_layers_0based,
        resolved_rescue_layer_0based,
        float(corruption_noise_scale),
        INTERVENTION_BATCH_SIZE,
        intervention_kind="corruption_only",
        desc=f"corruption only, scale {corruption_noise_scale}",
    ).rename(
        columns={
            "prediction_idx": "corrupted_prediction_idx",
            "correct_margin": "corrupted_correct_margin",
            "is_correct": "corrupted_is_correct",
        }
    )
    corrupted_part["corruption_noise_scale"] = float(corruption_noise_scale)
    corrupted_frames.append(corrupted_part)

corrupted_df = pd.concat(corrupted_frames, ignore_index=True)
corrupted_df = corrupted_df.merge(
    eval_clean_df[["example_id", "clean_prediction_idx", "clean_correct_margin", "clean_is_correct"]],
    on="example_id",
    how="left",
    validate="many_to_one",
)
corrupted_df["corruption_layer_1based"] = corrupted_df["corruption_layer"] + 1
corrupted_df["correct_margin_delta"] = corrupted_df["corrupted_correct_margin"] - corrupted_df["clean_correct_margin"]
corrupted_df["prediction_preserved"] = corrupted_df["corrupted_prediction_idx"].eq(corrupted_df["clean_prediction_idx"])
corrupted_df["clean_correct_broken"] = corrupted_df["clean_is_correct"] & (~corrupted_df["corrupted_is_correct"])

patch_frames = []
for corruption_noise_scale in CORRUPTION_NOISE_SCALES:
    patch_part = run_fixed_site_intervention_scan(
        eval_rows,
        resolved_corruption_layers_0based,
        resolved_rescue_layer_0based,
        float(corruption_noise_scale),
        INTERVENTION_BATCH_SIZE,
        intervention_kind="clean_patch",
        clean_rescue_hidden=eval_rescue_hidden,
        desc=f"clean patch, scale {corruption_noise_scale}",
    ).rename(
        columns={
            "prediction_idx": "patched_prediction_idx",
            "correct_margin": "patched_correct_margin",
            "is_correct": "patched_is_correct",
        }
    )
    patch_part["corruption_noise_scale"] = float(corruption_noise_scale)
    patch_frames.append(patch_part)

patch_df = pd.concat(patch_frames, ignore_index=True)
patch_df = patch_df.merge(
    corrupted_df[[
        "example_id",
        "corruption_layer",
        "corruption_layer_1based",
        "corruption_noise_scale",
        "clean_prediction_idx",
        "clean_correct_margin",
        "clean_is_correct",
        "corrupted_prediction_idx",
        "corrupted_correct_margin",
        "corrupted_is_correct",
        "clean_correct_broken",
    ]],
    on=["example_id", "corruption_layer", "corruption_noise_scale"],
    how="left",
    validate="one_to_one",
)
patch_df["rescue_layer_1based"] = resolved_rescue_layer_1based
patch_df["margin_recovery"] = patch_df["patched_correct_margin"] - patch_df["corrupted_correct_margin"]
patch_df["prediction_matches_clean"] = patch_df["patched_prediction_idx"].eq(patch_df["clean_prediction_idx"])
patch_df["broken_case_rescued"] = patch_df["clean_correct_broken"] & patch_df["patched_is_correct"]

steering_frames = []
for steering_method, steering_direction in steering_directions.items():
    for steering_scale in STEERING_SCALES:
        for corruption_noise_scale in CORRUPTION_NOISE_SCALES:
            steering_part = run_fixed_site_intervention_scan(
                eval_rows,
                resolved_corruption_layers_0based,
                resolved_rescue_layer_0based,
                float(corruption_noise_scale),
                INTERVENTION_BATCH_SIZE,
                intervention_kind="steering",
                steering_direction=steering_direction,
                steering_scale=steering_scale,
                desc=f"{steering_method}, steering scale {steering_scale}, corruption scale {corruption_noise_scale}",
            ).rename(
                columns={
                    "prediction_idx": "steered_prediction_idx",
                    "correct_margin": "steered_correct_margin",
                    "is_correct": "steered_is_correct",
                }
            )
            steering_part["steering_method"] = steering_method
            steering_part["steering_scale"] = float(steering_scale)
            steering_part["corruption_noise_scale"] = float(corruption_noise_scale)
            steering_frames.append(steering_part)

steering_df = pd.concat(steering_frames, ignore_index=True)
steering_df = steering_df.merge(
    corrupted_df[[
        "example_id",
        "corruption_layer",
        "corruption_layer_1based",
        "corruption_noise_scale",
        "clean_prediction_idx",
        "clean_correct_margin",
        "clean_is_correct",
        "corrupted_prediction_idx",
        "corrupted_correct_margin",
        "corrupted_is_correct",
        "clean_correct_broken",
    ]],
    on=["example_id", "corruption_layer", "corruption_noise_scale"],
    how="left",
    validate="many_to_one",
)
steering_df["rescue_layer_1based"] = resolved_rescue_layer_1based
steering_df["margin_recovery"] = steering_df["steered_correct_margin"] - steering_df["corrupted_correct_margin"]
steering_df["prediction_matches_clean"] = steering_df["steered_prediction_idx"].eq(steering_df["clean_prediction_idx"])
steering_df["broken_case_rescued"] = steering_df["clean_correct_broken"] & steering_df["steered_is_correct"]

display(corrupted_df.head())
display(patch_df.head())
display(steering_df.head())


## Basic Analysis


### Corruption-Only Summary


In [ ]:
corrupted_summary_by_scale = (
    corrupted_df.groupby(["corruption_noise_scale", "corruption_layer_1based"])
    .agg(
        mean_corrupted_correct_margin=("corrupted_correct_margin", "mean"),
        mean_correct_margin_delta=("correct_margin_delta", "mean"),
        prediction_preservation_rate=("prediction_preserved", "mean"),
        corrupted_accuracy=("corrupted_is_correct", "mean"),
        clean_correct_break_rate=("clean_correct_broken", "mean"),
    )
    .reset_index()
)

corrupted_summary = (
    corrupted_summary_by_scale.groupby("corruption_layer_1based")
    .agg(
        mean_correct_margin_delta_mean=("mean_correct_margin_delta", "mean"),
        mean_correct_margin_delta_std=("mean_correct_margin_delta", "std"),
        prediction_preservation_rate_mean=("prediction_preservation_rate", "mean"),
        prediction_preservation_rate_std=("prediction_preservation_rate", "std"),
        clean_correct_break_rate_mean=("clean_correct_break_rate", "mean"),
        clean_correct_break_rate_std=("clean_correct_break_rate", "std"),
    )
    .reset_index()
    .fillna(0.0)
)

display(corrupted_summary_by_scale.round(4))
display(corrupted_summary.round(4))


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 12), sharex=True)

x = corrupted_summary["corruption_layer_1based"].to_numpy()

axes[0].plot(x, corrupted_summary["mean_correct_margin_delta_mean"], marker="o")
axes[0].fill_between(
    x,
    corrupted_summary["mean_correct_margin_delta_mean"] - corrupted_summary["mean_correct_margin_delta_std"],
    corrupted_summary["mean_correct_margin_delta_mean"] + corrupted_summary["mean_correct_margin_delta_std"],
    alpha=0.2,
)
axes[0].axhline(0.0, color="black", linewidth=1, alpha=0.6)
axes[0].set_ylabel("Mean delta")
axes[0].set_title("Mean and standard deviation of correct-answer margin change across configured corruption scales")

axes[1].plot(x, corrupted_summary["prediction_preservation_rate_mean"], marker="o")
axes[1].fill_between(
    x,
    corrupted_summary["prediction_preservation_rate_mean"] - corrupted_summary["prediction_preservation_rate_std"],
    corrupted_summary["prediction_preservation_rate_mean"] + corrupted_summary["prediction_preservation_rate_std"],
    alpha=0.2,
)
axes[1].set_ylabel("Dataset fraction")
axes[1].set_title("Mean and standard deviation of prediction preservation across configured corruption scales")

axes[2].plot(x, corrupted_summary["clean_correct_break_rate_mean"], marker="o")
axes[2].fill_between(
    x,
    corrupted_summary["clean_correct_break_rate_mean"] - corrupted_summary["clean_correct_break_rate_std"],
    corrupted_summary["clean_correct_break_rate_mean"] + corrupted_summary["clean_correct_break_rate_std"],
    alpha=0.2,
)
axes[2].set_xlabel("Corruption layer")
axes[2].set_ylabel("Dataset fraction")
axes[2].set_title("Mean and standard deviation of clean-correct to incorrect rate across configured corruption scales")

plt.tight_layout()
plt.show()


### Fixed-Site Same-Example Clean Patching Summary


In [ ]:
patch_summary_by_scale = (
    patch_df.groupby(["corruption_noise_scale", "corruption_layer_1based"])
    .agg(
        mean_patched_correct_margin=("patched_correct_margin", "mean"),
        mean_margin_recovery=("margin_recovery", "mean"),
        patch_prediction_matches_clean_rate=("prediction_matches_clean", "mean"),
        patched_accuracy=("patched_is_correct", "mean"),
        broken_case_rescue_rate=("broken_case_rescued", "mean"),
    )
    .reset_index()
)

patch_summary = (
    patch_summary_by_scale.groupby("corruption_layer_1based")
    .agg(
        mean_margin_recovery_mean=("mean_margin_recovery", "mean"),
        mean_margin_recovery_std=("mean_margin_recovery", "std"),
        patch_prediction_matches_clean_rate_mean=("patch_prediction_matches_clean_rate", "mean"),
        patch_prediction_matches_clean_rate_std=("patch_prediction_matches_clean_rate", "std"),
        patched_accuracy_mean=("patched_accuracy", "mean"),
        patched_accuracy_std=("patched_accuracy", "std"),
        broken_case_rescue_rate_mean=("broken_case_rescue_rate", "mean"),
        broken_case_rescue_rate_std=("broken_case_rescue_rate", "std"),
    )
    .reset_index()
    .fillna(0.0)
)

display(patch_summary_by_scale.round(4))
display(patch_summary.round(4))


In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(12, 16), sharex=True)

x = patch_summary["corruption_layer_1based"].to_numpy()

axes[0].plot(x, patch_summary["mean_margin_recovery_mean"], marker="o")
axes[0].fill_between(
    x,
    patch_summary["mean_margin_recovery_mean"] - patch_summary["mean_margin_recovery_std"],
    patch_summary["mean_margin_recovery_mean"] + patch_summary["mean_margin_recovery_std"],
    alpha=0.2,
)
axes[0].axhline(0.0, color="black", linewidth=1, alpha=0.6)
axes[0].set_ylabel("Mean recovery")
axes[0].set_title("Mean and standard deviation of correct-answer margin recovery across configured corruption scales")

axes[1].plot(x, patch_summary["patch_prediction_matches_clean_rate_mean"], marker="o")
axes[1].fill_between(
    x,
    patch_summary["patch_prediction_matches_clean_rate_mean"] - patch_summary["patch_prediction_matches_clean_rate_std"],
    patch_summary["patch_prediction_matches_clean_rate_mean"] + patch_summary["patch_prediction_matches_clean_rate_std"],
    alpha=0.2,
)
axes[1].set_ylabel("Dataset fraction")
axes[1].set_title("Mean and standard deviation of clean-prediction match rate after fixed-site clean patching")

axes[2].plot(x, patch_summary["patched_accuracy_mean"], marker="o")
axes[2].fill_between(
    x,
    patch_summary["patched_accuracy_mean"] - patch_summary["patched_accuracy_std"],
    patch_summary["patched_accuracy_mean"] + patch_summary["patched_accuracy_std"],
    alpha=0.2,
)
axes[2].set_ylabel("Dataset fraction")
axes[2].set_title("Mean and standard deviation of patched-run accuracy across configured corruption scales")

axes[3].plot(x, patch_summary["broken_case_rescue_rate_mean"], marker="o")
axes[3].fill_between(
    x,
    patch_summary["broken_case_rescue_rate_mean"] - patch_summary["broken_case_rescue_rate_std"],
    patch_summary["broken_case_rescue_rate_mean"] + patch_summary["broken_case_rescue_rate_std"],
    alpha=0.2,
)
axes[3].set_xlabel("Earlier corruption layer")
axes[3].set_ylabel("Dataset fraction")
axes[3].set_title("Mean and standard deviation of rescue rate across configured corruption scales with fixed-site clean patching")

plt.tight_layout()
plt.show()


### Fixed-Site Steering Summary


In [ ]:
steering_summary_by_scale = (
    steering_df.groupby(["steering_method", "steering_scale", "corruption_noise_scale", "corruption_layer_1based"])
    .agg(
        mean_steered_correct_margin=("steered_correct_margin", "mean"),
        mean_margin_recovery=("margin_recovery", "mean"),
        steering_prediction_matches_clean_rate=("prediction_matches_clean", "mean"),
        steered_accuracy=("steered_is_correct", "mean"),
        broken_case_rescue_rate=("broken_case_rescued", "mean"),
    )
    .reset_index()
)

steering_summary = (
    steering_summary_by_scale.groupby(["steering_method", "steering_scale", "corruption_layer_1based"])
    .agg(
        mean_margin_recovery_mean=("mean_margin_recovery", "mean"),
        mean_margin_recovery_std=("mean_margin_recovery", "std"),
        steered_accuracy_mean=("steered_accuracy", "mean"),
        steered_accuracy_std=("steered_accuracy", "std"),
        broken_case_rescue_rate_mean=("broken_case_rescue_rate", "mean"),
        broken_case_rescue_rate_std=("broken_case_rescue_rate", "std"),
    )
    .reset_index()
    .fillna(0.0)
)

steering_average_summary = (
    steering_summary.groupby(["steering_method", "steering_scale"])[
        ["mean_margin_recovery_mean", "steered_accuracy_mean", "broken_case_rescue_rate_mean"]
    ]
    .mean()
    .reset_index()
)

display(steering_summary_by_scale.round(4))
display(steering_summary.round(4))
display(steering_average_summary.round(4))


In [ ]:
methods = steering_summary["steering_method"].drop_duplicates().tolist()
fig, axes = plt.subplots(len(methods), 2, figsize=(16, 5 * len(methods)), sharex=True)
if len(methods) == 1:
    axes = np.array([axes])

for row_idx, method in enumerate(methods):
    part = steering_summary.loc[steering_summary["steering_method"].eq(method)].sort_values(["steering_scale", "corruption_layer_1based"])
    for scale in sorted(part["steering_scale"].unique()):
        scale_part = part.loc[part["steering_scale"].eq(scale)].sort_values("corruption_layer_1based")
        x = scale_part["corruption_layer_1based"].to_numpy()
        axes[row_idx, 0].plot(
            x,
            scale_part["mean_margin_recovery_mean"],
            marker="o",
            label=f"steering scale={scale:g}",
        )
        axes[row_idx, 0].fill_between(
            x,
            scale_part["mean_margin_recovery_mean"] - scale_part["mean_margin_recovery_std"],
            scale_part["mean_margin_recovery_mean"] + scale_part["mean_margin_recovery_std"],
            alpha=0.2,
        )
        axes[row_idx, 1].plot(
            x,
            scale_part["broken_case_rescue_rate_mean"],
            marker="o",
            label=f"steering scale={scale:g}",
        )
        axes[row_idx, 1].fill_between(
            x,
            scale_part["broken_case_rescue_rate_mean"] - scale_part["broken_case_rescue_rate_std"],
            scale_part["broken_case_rescue_rate_mean"] + scale_part["broken_case_rescue_rate_std"],
            alpha=0.2,
        )

    axes[row_idx, 0].axhline(0.0, color="black", linewidth=1, alpha=0.6)
    axes[row_idx, 0].set_ylabel("Mean recovery")
    axes[row_idx, 0].set_title(f"Mean and standard deviation of margin recovery across configured corruption scales with {method.lower()}")
    axes[row_idx, 0].legend()

    axes[row_idx, 1].set_ylabel("Dataset fraction")
    axes[row_idx, 1].set_title(f"Mean and standard deviation of rescue rate across configured corruption scales with {method.lower()}")
    axes[row_idx, 1].legend()

axes[-1, 0].set_xlabel("Earlier corruption layer")
axes[-1, 1].set_xlabel("Earlier corruption layer")
plt.tight_layout()
plt.show()


In [ ]:
display(
    patch_summary.sort_values(
        ["broken_case_rescue_rate_mean", "mean_margin_recovery_mean"],
        ascending=[False, False],
    )
    .head(10)
    .round(4)
)

display(
    steering_average_summary.sort_values(
        ["broken_case_rescue_rate_mean", "mean_margin_recovery_mean"],
        ascending=[False, False],
    )
    .round(4)
)
